# Sentiment Analysis 

### Naive Bayes 
- Gaussian Naive Bayes (GaussianNB)
- Multinomial Naive Bayes (MultinomialNB)
- Bernoulli Naive Bayes (BernoulliNB)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.describe

<bound method NDFrame.describe of                                                   review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]>

In [5]:
df ["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

## Text data cleaning 
1. Sample 10000 rows 
2. Remove html tags 
3. Remove special characters 
4. Converting every thing to lower case 
5. Removing stop words 
6. Stamming


In [6]:
df= df.sample(10000)

In [7]:
df.info

<bound method DataFrame.info of                                                   review sentiment
25347  A funny and scathing critique of Russian socie...  positive
13222  A very positive message for our youth is shown...  positive
47391  This movie was recently shown subtitled not du...  positive
46589  Wow! I remember so many awful films that loose...  negative
31716  Cliffhanger is a decent action crime adventure...  positive
...                                                  ...       ...
29569  This movie was nominated for best picture but ...  positive
42034  Misguided, miscast, murky, interminable lunacy...  negative
33056  Billed as Takashi Miike's "first family film" ...  positive
24585  The odd mixture of comedy and horror sometimes...  negative
28552  This movie is an exact copy of a TV series on ...  positive

[10000 rows x 2 columns]>

In [8]:
# df['sentiment'].replace({"positive": 1 , "negative":0},inplace= True)  # it may automatic downcasting

df['sentiment'] = df['sentiment'].replace({
    "positive": 1,
    "negative": 0
})
# the second version is the recommended modern Pandas style.

/tmp/ipykernel_59/928976059.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({


In [44]:
df.head()

,review,sentiment
25347,a funni and scath critiqu of russian societi a...,1
13222,a veri posit messag for our youth is shown in ...,1
47391,thi movi wa recent shown subtitl not dub on au...,1
46589,wow i rememb so mani aw film that loos revolv ...,0
31716,cliffhang is a decent action crime adventur wi...,1


In [10]:
"""
Removing HTML tags from the review text using Regular Expressions (regex). The regex pattern '<.*?>' matches any substring that starts with '<', followed by any characters (non-greedy), and ends with '>'. This effectively captures HTML tags, which can then be removed from the text.

"""

import re # Regular Expression

clean  = re.compile('<.*?>')  # regex to remove html tags
re.sub(clean, '', df.iloc[2].review ) # "Take the review in the 3rd row, find all HTML tags, and replace them with nothing."

'This movie was recently shown subtitled not dubbed, on Australia\'s Special Broadcasting Service. I thought I read here that there is sometimes an assumption that *Revenge of the Rats* is a sequel. In fact, the rats are getting revenge for something that is done within this movie, or in human history. There is no prequel as far as I know.Now, let\'s get The Pied Piper of Hamelin out of the way. Perhaps this tale crossed my mind when I was watching the film, but that\'s all. There was the use of the word "coffers" at the end of the movie. A dazed mayor tries to justify her actions by saying "the coffers were empty". "Coffers" isn\'t a word you hear around much these days, unless you read folk tales to your children, as I do. Since seeing *Revenge of the Rats*, I\'ve read two versions of The Pied Piper of Hamelin. Now I see the comparison. A metaphorical fat but literally ugly mayor promises more than the city can afford, to get rid of the rats. This is about as far as I can follow the 

In [11]:
# function to clean the review text by removing HTML tags 
def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)  

In [12]:
df['review'] = df['review'].apply(clean_html)

In [13]:
# conver lower case
def convert_lower(text):
    return text.lower()

In [14]:
df['review'] = df['review'].apply(convert_lower)

In [15]:
# function to remove special characters from the review text using Regular Expressions (regex). The regex pattern '[^a-zA-Z0-9\s]' matches any character that is not a letter (uppercase or lowercase), a digit, or whitespace. This effectively captures special characters, which can then be removed from the text.
def remove_special_characters(text):
    x=''
    for i in text:
        if i.isalnum() or i.isspace():
            x+=i
        else:
            x+=' '

    return x

In [16]:
df['review'] = df['review'].apply(remove_special_characters)

In [17]:
# remove stop words from the review text using the Natural Language Toolkit (nltk). Stop words are common words that are often removed from text data to improve the performance of natural language processing models. The function takes a string of text as input, splits it into individual words, filters out any stop words, and then joins the remaining words back into a single string.
import nltk
from nltk.corpus import stopwords
def remove_stop_words(text):
    x=[]
    for i in text.split():
        if i in stopwords.words('english'):
            x.append(i)

    y=x[:]
    x.clear()
    return y

In [18]:
df['review'] = df['review'].apply(remove_special_characters)

In [19]:
df["review"] = df["review"].str.split()
df["review"].iloc[0]


['a',
 'funny',
 'and',
 'scathing',
 'critique',
 'of',
 'russian',
 'society',
 'and',
 'culture',
 'during',
 'the',
 'transition',
 'from',
 'communism',
 'okno',
 'v',
 'parizh',
 'also',
 'shows',
 'the',
 'west',
 'in',
 'an',
 'unfavorable',
 'light',
 'a',
 'group',
 'of',
 'russians',
 'living',
 'in',
 'st',
 'petersburg',
 'a',
 'k',
 'a',
 'peter',
 'the',
 'great',
 's',
 'window',
 'on',
 'the',
 'west',
 'find',
 'a',
 'magic',
 'portal',
 'that',
 'instantly',
 'transports',
 'them',
 'to',
 'paris',
 'mamin',
 's',
 'film',
 'is',
 'truly',
 'hilarious',
 'and',
 'just',
 'weird',
 'enough',
 'constantly',
 'keep',
 'even',
 'the',
 'jaded',
 'film',
 'viewer',
 'on',
 'his',
 'toes',
 'the',
 'songs',
 'the',
 'dream',
 'sequences',
 'and',
 'the',
 'deliciously',
 'disgusting',
 'fringes',
 'of',
 'society',
 'from',
 'both',
 'cultures',
 'mingle',
 'to',
 'create',
 'a',
 'memorable',
 'and',
 'meaningful',
 'film',
 'anyone',
 'trying',
 'to',
 'understand',
 'th

In [20]:
df.head()

,review,sentiment
25347,"[a, funny, and, scathing, critique, of, russia...",1
13222,"[a, very, positive, message, for, our, youth, ...",1
47391,"[this, movie, was, recently, shown, subtitled,...",1
46589,"[wow, i, remember, so, many, awful, films, tha...",0
31716,"[cliffhanger, is, a, decent, action, crime, ad...",1


In [21]:
# Performing sentiment analysis on the cleaned review text using the TextBlob library. The function takes a string of text as input, creates a TextBlob object, and then calculates the sentiment polarity of the text. The polarity score ranges from -1 (negative sentiment) to 1 (positive sentiment), with 0 indicating neutral sentiment. The function returns the polarity score as a float.
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [22]:
y=[]
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z= y[:]
    y.clear()
    return z

In [23]:
stem_words(["I","loved", "loving","it"])

['i', 'love', 'love', 'it']

In [24]:
df['review'] = df['review'].apply(stem_words)

In [25]:
# join back
def join_back(text):
    return ' '.join(text)

In [26]:
df['review'] = df['review'].apply(join_back)

In [27]:
df.review.head()

25347    a funni and scath critiqu of russian societi a...
13222    a veri posit messag for our youth is shown in ...
47391    thi movi wa recent shown subtitl not dub on au...
46589    wow i rememb so mani aw film that loos revolv ...
31716    cliffhang is a decent action crime adventur wi...
Name: review, dtype: object

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [29]:
X=cv.fit_transform(df['review']).toarray()

In [30]:
X.shape

(10000, 36204)

In [31]:
y = df.iloc[:, -1].values 

In [32]:
y.shape

(10000,)

In [33]:
y

array([1, 1, 1, ..., 1, 0, 1])

In [34]:
# Split the dataset into training and testing sets using sklearn's train_test_split function.
# The test size is set to 20% of the dataset, and a random state is provided for reproducibility.
from sklearn.model_selection import train_test_split

In [35]:
X_train , X_test , y_train , y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [36]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 36204), (2000, 36204), (8000,), (2000,))

In [37]:
from sklearn.naive_bayes import GaussianNB ,MultinomialNB, BernoulliNB

In [38]:
clf1 = GaussianNB()
clf2 = MultinomialNB()
clf3 = BernoulliNB()

In [39]:
clf1.fit(X_train , y_train)

GaussianNB()

In [40]:
clf2.fit(X_train , y_train)

MultinomialNB()

In [41]:
clf3.fit(X_train , y_train)

BernoulliNB()